# Freeze the selected CatBoost baseline pipeline

E3 selection uses the completed seven-configuration Validation A comparison. DummyClassifier is excluded from selectable learned configurations. This notebook copies and verifies the already-fitted selected pipeline; it does not fit another model or predict Validation B/Test. Use Restart Kernel and Run All to create a new immutable selection record.

## Environment and repository

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import importlib.metadata
import json
import platform
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

current = Path.cwd().resolve()
ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "docs/predictor_list.csv").is_file()
     and (p / "notebooks").is_dir()), None
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project repository.")
print("Repository:", ROOT)
print("Python:", platform.python_version())


Repository: C:\Users\LENOVO\OneDrive\Desktop\PTTK
Python: 3.14.6


## Identify the completed comparison

Use an explicit source rather than implicitly changing the choice when a newer experiment appears.

In [2]:
import shutil
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss

COMPARISON_PATH = ROOT / "outputs/tables/machine_learning/ml_catboost_weighted/20261111_153704_495324/seven_model_comparison.csv"
comparison = pd.read_csv(COMPARISON_PATH)
assert len(comparison) == 7 and comparison.model.is_unique
assert comparison.split.eq("validation_a").all()
assert comparison.n.eq(7661).all() and comparison.late.eq(387).all()
assert np.isfinite(comparison[["AP", "ROC_AUC", "Brier", "LogLoss"]].to_numpy()).all()
source_map = json.loads((COMPARISON_PATH.parent / "comparison_source.json").read_text(encoding="utf-8"))
display(comparison)


,model,split,probability_type,n,late,prevalence,AP,ROC_AUC,Brier,LogLoss
0,CatBoost,validation_a,raw,7661,387,0.050516,0.092855,0.655757,0.047227,0.192596
1,LogisticRegression,validation_a,raw,7661,387,0.050516,0.091637,0.642369,0.047720,0.196408
2,CatBoostWeighted,validation_a,raw,7661,387,0.050516,0.079147,0.602951,0.159419,0.499574
3,XGBoost,validation_a,raw,7661,387,0.050516,0.076845,0.596653,0.047978,0.199795
4,RandomForest,validation_a,raw,7661,387,0.050516,0.074766,0.604052,0.048191,0.200893
5,LightGBM,validation_a,raw,7661,387,0.050516,0.073623,0.582513,0.048189,0.201357
6,DummyClassifier,validation_a,raw,7661,387,0.050516,0.050516,0.500000,0.048166,0.201833


## Apply the predeclared selection rule

Highest AP among learned configurations; AP ties within 1e-12 use lower Log Loss, lower Brier, then unweighted, then configuration ID. Assert the chosen model is the requested CatBoost baseline.

In [3]:
candidates = comparison.loc[comparison.model.ne("DummyClassifier")].copy()
candidates["weighted"] = candidates.model.eq("CatBoostWeighted")
candidates["config_id"] = candidates.model
best_ap = candidates.AP.max()
tied = candidates.loc[(candidates.AP - best_ap).abs().le(1e-12)]
winner = tied.sort_values(["LogLoss", "Brier", "weighted", "config_id"]).iloc[0]
assert winner.model == "CatBoost", "The protocol no longer selects CatBoost baseline; inspect the comparison."
source_run = ROOT / source_map["ml_catboost_dummy"]
source_model = ROOT / "outputs/models/ml_catboost_dummy" / source_run.name / "CatBoost.joblib"
source_plan = json.loads((source_run / "experiment_plan.json").read_text(encoding="utf-8"))
assert source_plan["weighting"] == "none"
assert source_plan["CatBoost"].get("class_weights") is None
print("Selected:", winner.model)
print("Source pipeline:", source_model.relative_to(ROOT))


Selected: CatBoost
Source pipeline: outputs\models\ml_catboost_dummy\20261111_152855_922857\CatBoost.joblib


## Verify source data and experiment lineage

Validate all previous model groups against current file hashes. Only Train and Validation A are loaded for prediction verification; no B/Test probabilities are generated.

In [4]:
INPUTS = [
    "data/processed/research_data_weather_price_holiday_assume_utc.csv",
    "docs/predictor_list.csv",
    "data/ml_preparation/split_manifest.csv",
]
hashes = {name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in INPUTS}
lf_hashes = {name: hashlib.sha256((ROOT / name).read_bytes().replace(b"\r\n", b"\n")).hexdigest() for name in INPUTS}
for directory in [COMPARISON_PATH.parent, *[ROOT / value for value in source_map.values()]]:
    recorded = {k.replace("\\", "/"): v for k, v in json.loads((directory / "input_hashes.json").read_text(encoding="utf-8")).items()}
    assert recorded == hashes or recorded == lf_hashes, directory

features = pd.read_csv(ROOT / INPUTS[1]).feature.tolist()
assert len(features) == len(set(features)) == 21
data = pd.read_csv(ROOT / INPUTS[0])
manifest = pd.read_csv(ROOT / INPUTS[2])
assert data.order_id.is_unique and manifest.order_id.is_unique
selected_orders = manifest.loc[manifest.split.isin(["train", "validation_a"]), ["order_id", "split"]]
joined = selected_orders.merge(data, on="order_id", how="left", validate="one_to_one", indicator=True)
assert joined["_merge"].eq("both").all()
train = joined.loc[joined.split.eq("train")].copy()
val = joined.loc[joined.split.eq("validation_a")].copy()
assert len(train) == 57449 and train.is_late.eq(1).sum() == 3720
assert len(val) == 7661 and val.is_late.eq(1).sum() == 387
cat = ["customer_state", "purchase_weekday"]
num = [name for name in features if name not in cat]
for frame in [train, val]:
    for name in cat:
        frame[name] = frame[name].map(lambda value: str(value) if pd.notna(value) else np.nan)
    for name in num:
        frame[name] = pd.to_numeric(frame[name], errors="raise").replace([np.inf, -np.inf], np.nan)


## Verify the fitted pipeline on Validation A

Reload the existing model, check its parameters and Train medians, then reproduce its saved validation probabilities and metrics. No refitting occurs.

In [5]:
source_versions = json.loads((source_run / "package_versions.json").read_text(encoding="utf-8"))
current_versions = {name: importlib.metadata.version(name) for name in source_versions if name != "python"}
assert all(current_versions[name] == version for name, version in source_versions.items() if name != "python"), "Environment differs from the selected run."
selected_pipeline = joblib.load(source_model)
classifier = selected_pipeline.named_steps["classifier"]
assert classifier.get_params() == source_plan["CatBoost"]
assert classifier.tree_count_ == 100
np.testing.assert_allclose(
    selected_pipeline.named_steps["preprocessing"].named_transformers_["numeric"].statistics_,
    train[num].median().to_numpy(),
)
probabilities = selected_pipeline.predict_proba(val[features])[:, 1]
assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
recorded_predictions = pd.read_csv(source_run / "validation_predictions.csv", float_precision="round_trip")
recorded_predictions = recorded_predictions.loc[recorded_predictions.model.eq("CatBoost")].set_index("order_id")
assert recorded_predictions.index.is_unique and len(recorded_predictions) == len(val)
recorded_predictions = recorded_predictions.loc[val.order_id]
np.testing.assert_array_equal(recorded_predictions.y_true.to_numpy(), val.is_late.astype(int).to_numpy())
np.testing.assert_allclose(probabilities, recorded_predictions.probability.to_numpy(), rtol=1e-12, atol=1e-14)
verified_metrics = dict(
    AP=average_precision_score(val.is_late, probabilities),
    ROC_AUC=roc_auc_score(val.is_late, probabilities),
    Brier=brier_score_loss(val.is_late, probabilities),
    LogLoss=log_loss(val.is_late, probabilities, labels=[0, 1]),
)
np.testing.assert_allclose(
    list(verified_metrics.values()), winner[["AP", "ROC_AUC", "Brier", "LogLoss"]].to_numpy(dtype=float),
    rtol=1e-8, atol=1e-10,
)
display(pd.DataFrame([verified_metrics]))


,AP,ROC_AUC,Brier,LogLoss
0,0.092855,0.655757,0.047227,0.192596


## Freeze pipeline and selected_config.json

Create new selection artifacts without modifying the original fitted run. Null calibration/threshold fields explicitly identify the remaining work.

In [6]:
selection_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
tables = ROOT / "outputs/tables/machine_learning/ml_selection" / selection_id
models = ROOT / "outputs/models/ml_selection" / selection_id
for directory in [tables, models]:
    directory.mkdir(parents=True, exist_ok=False)
frozen_model = models / "CatBoostBaseline.joblib"
shutil.copy2(source_model, frozen_model)
model_hash = hashlib.sha256(frozen_model.read_bytes()).hexdigest()
assert model_hash == hashlib.sha256(source_model.read_bytes()).hexdigest()
np.testing.assert_allclose(
    joblib.load(frozen_model).predict_proba(val[features])[:, 1], probabilities,
    rtol=1e-12, atol=1e-14,
)
comparison.to_csv(tables / "selection_comparison.csv", index=False)
recorded_predictions.reset_index().to_csv(tables / "validation_a_predictions.csv", index=False)
configuration = {
    "selection_id": selection_id,
    "selected_model": "CatBoost", "configuration": "unweighted_baseline",
    "status": "classifier_frozen_pending_calibration",
    "selection_split": "validation_a", "fit_split": "train",
    "selection_rule": "max AP; ties <=1e-12: lower LogLoss, lower Brier, unweighted, configuration ID",
    "selection_reason": "Highest Validation A AP among six learned configurations; weighted CatBoost was worse on all four metrics.",
    "source_run": source_run.relative_to(ROOT).as_posix(),
    "source_model": source_model.relative_to(ROOT).as_posix(),
    "comparison_source": COMPARISON_PATH.relative_to(ROOT).as_posix(),
    "comparison_sha256": hashlib.sha256(COMPARISON_PATH.read_bytes()).hexdigest(),
    "frozen_pipeline": frozen_model.relative_to(ROOT).as_posix(), "pipeline_sha256": model_hash,
    "features": features, "categorical_features": cat,
    "parameters": classifier.get_params(), "fitted_parameters": classifier.get_all_params(),
    "weighting": "none", "train_orders": len(train), "train_late": int(train.is_late.sum()),
    "validation_a_orders": len(val), "validation_a_late": int(val.is_late.sum()),
    "validation_a_metrics": verified_metrics,
    "input_hashes": hashes, "input_lf_hashes": lf_hashes,
    "source_package_versions": source_versions,
    "verification_package_versions": {**current_versions, "python": platform.python_version()},
    "refitted": False, "calibrator": None, "threshold": None,
    "validation_b_predicted": False, "test_predicted": False,
    "next_step": "E4 sigmoid calibration and threshold selection on Validation B",
    "limitations": "Validation A reused for exploratory selection; AP lead over Logistic Regression has no uncertainty assessment. Test outcomes appeared previously in EDA/Stage 3.",
}
config_path = tables / "selected_config.json"
config_path.write_text(json.dumps(configuration, indent=2, default=str), encoding="utf-8")
for name, digest in hashes.items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest
print("Frozen pipeline:", frozen_model.relative_to(ROOT))
print("Selection record:", config_path.relative_to(ROOT))
print("No refitting, calibration, threshold selection or B/Test predictions performed.")


Frozen pipeline: outputs\models\ml_selection\20261111_154450_762570\CatBoostBaseline.joblib
Selection record: outputs\tables\ml_selection\20261111_154450_762570\selected_config.json
No refitting, calibration, threshold selection or B/Test predictions performed.
